In [41]:
import numpy as np
import pandas as pd 
import torch
import torch.nn as nn
df = pd.read_csv(r"D:\transformer architeture\english_to_bangla.csv")
df.head(10)

,en,bn
0,a child in a pink dress is climbing up a set o...,একটি গোলাপী জামা পরা বাচ্চা মেয়ে একটি বাড়ির প্...
1,a girl going into a wooden building .,একটি মেয়ে শিশু একটি কাঠের বাড়িতে ঢুকছে
2,a little girl climbing into a wooden playhouse .,একটি বাচ্চা তার কাঠের খেলাঘরে উঠছে ।
3,a little girl climbing the stairs to her playh...,ছোট মেয়েটি তার খেলার ঘরের সিড়ি বেয়ে উঠছে
4,a little girl in a pink dress going into a woo...,গোলাপি জামা পড়া ছোট একটি মেয়ে একটি কাঠের তৈরি...
5,a black dog and a spotted dog are fighting,একটি কালো কুকুর এবং একটি ছোপওয়ালা কুকুর ঝগড়া করছে
6,a black dog and a tri-colored dog playing with...,একটি কালো কুকুর একটি তিন রঙা কুকুরের সাথে রাস্...
7,a black dog and a white dog with brown spots a...,একটি কালো কুকুর ও একটি সাদা-বাদামি ছোপযুক্ত কু...
8,two dogs of different breeds looking at each o...,ভিন্ন জাতের দুটি কুকুর রাস্তায় একে অপরের দিকে...
9,two dogs on pavement moving toward each other .,রাস্তার পাশে দুইটি কুকুর পরস্পরের দিকে এগিয়ে য...


TOKENIZATION :-SPLITE AND TITLE PART 


In [42]:
def tokenization(text):
    new_text = []

    for sentence in text.dropna():
        words = sentence.upper().split()

        new_words = []

        for i in words:
            new_words.append(i.title())

        new_text.append(new_words)

    return new_text

df["tokens"] = tokenization(df["en"])

In [43]:


import torch
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [44]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Model loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda


In [45]:
texts = df["tokens"].fillna("").astype(str).tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1221 [00:00<?, ?it/s]

Embedding shape: (39065, 384)


In [46]:
print( embeddings.shape[-1])

384


In [47]:
import torch
import torch.nn as nn
import math

d_model = embeddings.shape[1]
pos = 1

pe = torch.zeros(d_model)

for i in range(d_model):

    if i % 2 == 0:
        pe[i] = math.sin(
            pos / (10000 ** (i / d_model))
        )

    else:
        pe[i] = math.cos(
            pos / (10000 ** ((i - 1) / d_model))
        )

print("Positional Encoding:", pe)
print("Shape:", pe.shape)

Positional Encoding: tensor([8.4147e-01, 5.4030e-01, 8.1525e-01, 5.7911e-01, 7.8859e-01, 6.1492e-01,
        7.6172e-01, 6.4791e-01, 7.3482e-01, 6.7826e-01, 7.0806e-01, 7.0615e-01,
        6.8156e-01, 7.3176e-01, 6.5544e-01, 7.5524e-01, 6.2980e-01, 7.7676e-01,
        6.0469e-01, 7.9646e-01, 5.8019e-01, 8.1448e-01, 5.5634e-01, 8.3095e-01,
        5.3317e-01, 8.4601e-01, 5.1070e-01, 8.5976e-01, 4.8896e-01, 8.7231e-01,
        4.6795e-01, 8.8376e-01, 4.4767e-01, 8.9420e-01, 4.2813e-01, 9.0372e-01,
        4.0931e-01, 9.1240e-01, 3.9121e-01, 9.2030e-01, 3.7381e-01, 9.2750e-01,
        3.5711e-01, 9.3406e-01, 3.4108e-01, 9.4003e-01, 3.2571e-01, 9.4547e-01,
        3.1098e-01, 9.5042e-01, 2.9687e-01, 9.5492e-01, 2.8336e-01, 9.5901e-01,
        2.7043e-01, 9.6274e-01, 2.5806e-01, 9.6613e-01, 2.4623e-01, 9.6921e-01,
        2.3492e-01, 9.7201e-01, 2.2411e-01, 9.7456e-01, 2.1378e-01, 9.7688e-01,
        2.0391e-01, 9.7899e-01, 1.9449e-01, 9.8091e-01, 1.8549e-01, 9.8265e-01,
        1.7689e-01,

In [54]:
import torch
import torch.nn as nn
import math

torch.manual_seed(1200)

d_model = 384
num_heads = 12
seq_len = 60
batch_size = 8

# ---- Embeddings + positional encoding ----
embeddings = torch.as_tensor(embeddings, dtype=torch.float32)   # (60, 384)

final_token = embeddings + pe                                   # (60, 384)
print("Final Shape (before batch):", final_token.shape)

# Add batch dimension -> (8, 60, 384)
final_token = final_token.unsqueeze(0).repeat(batch_size, 1, 1)
print("Final Shape (with batch):", final_token.shape)


# ---- Q, K, V projection ----
class CreateMatrix(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        # scaled init avoids huge attention scores (randn without scaling saturates softmax)
        self.q = nn.Parameter(torch.randn(d_model, d_model) / math.sqrt(d_model))
        self.k = nn.Parameter(torch.randn(d_model, d_model) / math.sqrt(d_model))
        self.v = nn.Parameter(torch.randn(d_model, d_model) / math.sqrt(d_model))
        self.d_model = d_model

    def forward(self, x):
        Q = torch.matmul(x, self.q)
        K = torch.matmul(x, self.k)
        V = torch.matmul(x, self.v)
        return Q, K, V


# ---- Scaled dot-product attention ----
class SelfAttention(nn.Module):
    def forward(self, Q, K, V):
        dk = K.shape[-1]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(dk)
        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        return output, attention_weights


# ---- Multi-head attention ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.create_matrix = CreateMatrix(d_model)
        self.attention = SelfAttention()
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        Q, K, V = self.create_matrix(x)

        # (B, T, D) -> (B, H, T, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        output, attention_weights = self.attention(Q, K, V)

        # (B, H, T, head_dim) -> (B, T, D)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        output = self.W_o(output)
        return output, attention_weights


# ---- Run it ----
mha = MultiHeadAttention(d_model, num_heads)
# Use only the first 60 sentence embeddings; otherwise attention creates a huge 39065×39065 matrix.
final_token = final_token[:, :seq_len, :].to(device)
mha = mha.to(device)

output, attention_weights = mha(final_token)

print("Output shape:", output)                       # (8, 60, 384)
print("Attention weights shape:", attention_weights.shape) # (8, 12, 60, 60)

Final Shape (before batch): torch.Size([39065, 384])
Final Shape (with batch): torch.Size([8, 39065, 384])
Output shape: tensor([[[ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         ...,
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4450,  0.2929,  1.1720]],

        [[ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         ...,
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4449,  0.2929,  1.1720],
         [ 0.4885, -0.2554,  0.9325,  ..., -0.4450,  0.2929,  1.1720

In [55]:
embedding = nn.Embedding(39065, d_model)
seq_len = 60
input_ids = torch.randint(0, 39065, (8, seq_len))
final_token = embedding(input_ids)          # [8, 60, 384]

mha = MultiHeadAttention(d_model, num_heads)
output, attention_weights = mha(final_token)  # [8, 60, 384]  <-- this was missing
print(output)
class AddNorm(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, final_token, output):
        residual = final_token + output
        return self.layer_norm(residual)

add_norm = AddNorm(384)

final_output = add_norm(final_token, output)

print("output", output)
#print("final_token", final_token.shape)
#print("final_output", final_output.shape)  # torch.Size([2, 5, 384])
print("final_output", final_output) 

tensor([[[ 0.0952,  0.2890,  0.0453,  ..., -0.0118, -0.0113,  0.0726],
         [-0.0651,  0.1862, -0.2021,  ..., -0.0288,  0.0494, -0.0179],
         [-0.2856,  0.1629, -0.0667,  ..., -0.0332, -0.0768,  0.1028],
         ...,
         [-0.0738,  0.1458,  0.0955,  ...,  0.0708,  0.0073,  0.0478],
         [ 0.1861,  0.0180, -0.1577,  ..., -0.0058, -0.0233,  0.1403],
         [ 0.0887,  0.2289,  0.2428,  ..., -0.0399,  0.0673,  0.0720]],

        [[ 0.0529, -0.2121, -0.1803,  ...,  0.0247, -0.1216, -0.1550],
         [ 0.0369,  0.0381, -0.1718,  ...,  0.0173,  0.0008, -0.0722],
         [-0.1686,  0.0348, -0.0647,  ..., -0.0743, -0.1328,  0.2891],
         ...,
         [ 0.0765, -0.1515, -0.0791,  ...,  0.0915, -0.0495,  0.1168],
         [ 0.1576,  0.1462, -0.0364,  ...,  0.0319, -0.1377, -0.2235],
         [-0.0240,  0.0432, -0.1415,  ...,  0.0251, -0.1121,  0.0840]],

        [[-0.3167,  0.1172, -0.0834,  ...,  0.1210,  0.0990, -0.0441],
         [-0.3536,  0.0484, -0.1426,  ...,  0

In [ ]:

class FeedForward(nn.Module):

    def __init__(self, output):
        super().__init__()

        self.linear1 = nn.Linear(output, d_ff)

        self.relu = nn.GELU()

        self.linear2 = nn.Linear(d_ff, output)

    def forward(self, X):

        output = self.linear1(X)

        output = self.relu(output)

        output = self.linear2(output)

        return output
add_norm=AddNorm(384)
final_output1 = add_norm(final_output, final_output)
print(final_output1)


tensor([[[-0.5863, -0.9025, -0.3312,  ...,  2.7353, -0.3622,  0.0683],
         [ 0.2395,  0.7724,  0.0192,  ..., -0.6151,  0.5283, -1.0211],
         [ 0.1064, -1.9198, -1.2951,  ...,  0.2775, -2.5764,  0.2791],
         ...,
         [ 0.6919, -0.9735, -1.0495,  ...,  0.4526,  1.1074, -0.8154],
         [-0.4045, -0.1752, -0.8962,  ..., -1.9039, -1.1774,  0.5577],
         [-0.1595,  0.4572,  0.9205,  ...,  0.6141, -0.5199,  0.1192]],

        [[ 1.4231, -1.7309, -1.6634,  ...,  1.9429, -1.7352,  0.4877],
         [ 0.0052,  0.0068, -0.4423,  ..., -0.6791,  0.6962,  0.9698],
         [-0.2643,  2.6167, -0.8469,  ...,  0.3159, -0.8760, -0.5595],
         ...,
         [-1.4372,  0.6221,  0.1513,  ...,  0.5338,  0.8561, -1.3958],
         [ 0.7115, -0.6590, -0.3883,  ..., -2.1792, -1.0687,  1.0582],
         [ 1.2102,  0.8767, -0.4181,  ...,  0.0427,  0.2228,  1.8190]]],
       grad_fn=<NativeLayerNormBackward0>)


In [ ]:
class AddNorm1(nn.Module):

    def __init__(self, final_output1):
        super().__init__()

        self.layer_norm = nn.LayerNorm(final_output1)

    def forward(self, final_output, final_output1):

        # Residual connection
        residual = final_output+final_output1

        # Layer normalization
        output = self.layer_norm(residual)

        return output
add=AddNorm1(384)
final_output2=add(final_output,final_output1)
print(final_output2)

tensor([[[ 1.3024e+00, -1.3042e+00, -4.1758e-01,  ...,  9.5805e-01,
           9.7314e-01,  6.9140e-01],
         [ 1.4277e+00, -1.9036e+00,  1.5701e+00,  ...,  7.6454e-01,
           8.3954e-01,  1.0444e-01],
         [-9.5197e-01, -1.5794e+00,  1.7906e+00,  ...,  3.0569e-01,
           7.6617e-01,  1.1857e+00],
         [ 8.2916e-01, -2.1742e-02,  1.4889e+00,  ..., -8.4604e-01,
           6.7723e-01,  5.5501e-01],
         [-4.1684e-01,  1.1808e+00, -1.0856e-01,  ..., -1.2103e+00,
           1.5097e+00, -2.2238e-01]],

        [[ 1.8847e+00,  1.2193e+00, -2.6191e-01,  ..., -3.3010e-02,
          -1.1721e+00, -1.3958e+00],
         [-6.5857e-02,  5.8618e-01, -1.4566e+00,  ...,  2.2898e-01,
          -1.6794e+00,  5.9130e-01],
         [ 1.0142e+00,  3.1435e+00, -4.4797e-01,  ..., -7.9749e-01,
          -9.6955e-01, -1.1688e+00],
         [ 3.8683e-01,  1.5382e+00,  2.8214e-03,  ..., -6.6414e-01,
          -1.6692e+00,  8.9844e-02],
         [-6.1087e-01,  1.6034e+00,  1.2188e+00,  ...

In [ ]:
"""class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        max_len=100,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.pos_embedding = nn.Embedding(max_len, d_model)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask):

        B, S = src.shape

        pos = torch.arange(S, device=src.device).unsqueeze(0)

        X = self.embedding(src) + self.pos_embedding(pos)

        X = self.dropout(X)

        for layer in self.layers:

            X = layer(X, src_mask)

        return X"""

'class Encoder(nn.Module):\n\n    def __init__(\n        self,\n        vocab_size,\n        d_model,\n        num_heads,\n        d_ff,\n        num_layers,\n        max_len=100,\n        dropout=0.1\n    ):\n        super().__init__()\n\n        self.embedding = nn.Embedding(vocab_size, d_model)\n\n        self.pos_embedding = nn.Embedding(max_len, d_model)\n\n        self.layers = nn.ModuleList([\n            EncoderLayer(d_model, num_heads, d_ff, dropout)\n            for _ in range(num_layers)\n        ])\n\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, src, src_mask):\n\n        B, S = src.shape\n\n        pos = torch.arange(S, device=src.device).unsqueeze(0)\n\n        X = self.embedding(src) + self.pos_embedding(pos)\n\n        X = self.dropout(X)\n\n        for layer in self.layers:\n\n            X = layer(X, src_mask)\n\n        return X'

ENCODER PART OF TRANSFORMER FROM SCRATCH

In [ ]:
def tokenization(text):
    new_text = []

    for sentence in text.dropna():
        words = sentence.upper().split()

        new_words = []

        for i in words:
            new_words.append(i.title())

        new_text.append(new_words)

    return new_text

df["tokens_bn"] = tokenization(df["bn"])
df["tokens_bn"].head()

0    [একটি, গোলাপী, জামা, পরা, বাচ্চা, মেয়ে, একটি, ...
1       [একটি, মেয়ে, শিশু, একটি, কাঠের, বাড়িতে, ঢুকছে]
2         [একটি, বাচ্চা, তার, কাঠের, খেলাঘরে, উঠছে, ।]
3    [ছোট, মেয়েটি, তার, খেলার, ঘরের, সিড়ি, বেয়ে, উঠছে]
4    [গোলাপি, জামা, পড়া, ছোট, একটি, মেয়ে, একটি, কা...
Name: tokens_bn, dtype: object

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Model loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda


In [ ]:
texts1 = df["tokens_bn"].fillna("").astype(str).tolist()

embedding1 = model.encode(
    texts1,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embedding1.shape)


Batches:   0%|          | 0/1221 [00:00<?, ?it/s]

Embedding shape: (39065, 384)


In [ ]:
print(embedding1)

[[ 0.02832089 -0.08421122 -0.04360938 ...  0.06864612  0.06617825
   0.06730416]
 [ 0.02277156 -0.07031917 -0.02407127 ...  0.05409107  0.06283261
   0.04811767]
 [ 0.02791413 -0.02428371 -0.04874953 ...  0.07063959  0.05040115
   0.02947365]
 ...
 [ 0.03717348 -0.07664693 -0.04525765 ...  0.05630802  0.06084723
   0.05293587]
 [ 0.02424392 -0.07091185 -0.06804735 ...  0.05172296  0.06959698
   0.03120312]
 [ 0.02679316 -0.08325816 -0.03457869 ...  0.07434969  0.05447606
   0.04122701]]


In [ ]:
import torch
import torch.nn as nn
import math

d_model = 384
pos = 1

pe = torch.zeros(d_model)

for i in range(d_model):

    if i % 2 == 0:
        pe[i] = math.sin(
            pos / (10000 ** (i / d_model))
        )

    else:
        pe[i] = math.cos(
            pos / (10000 ** ((i - 1) / d_model))
        )

print("Positional Encoding:", pe)
#print("Shape:", pe.shape)

Positional Encoding: tensor([8.4147e-01, 5.4030e-01, 8.1525e-01, 5.7911e-01, 7.8859e-01, 6.1492e-01,
        7.6172e-01, 6.4791e-01, 7.3482e-01, 6.7826e-01, 7.0806e-01, 7.0615e-01,
        6.8156e-01, 7.3176e-01, 6.5544e-01, 7.5524e-01, 6.2980e-01, 7.7676e-01,
        6.0469e-01, 7.9646e-01, 5.8019e-01, 8.1448e-01, 5.5634e-01, 8.3095e-01,
        5.3317e-01, 8.4601e-01, 5.1070e-01, 8.5976e-01, 4.8896e-01, 8.7231e-01,
        4.6795e-01, 8.8376e-01, 4.4767e-01, 8.9420e-01, 4.2813e-01, 9.0372e-01,
        4.0931e-01, 9.1240e-01, 3.9121e-01, 9.2030e-01, 3.7381e-01, 9.2750e-01,
        3.5711e-01, 9.3406e-01, 3.4108e-01, 9.4003e-01, 3.2571e-01, 9.4547e-01,
        3.1098e-01, 9.5042e-01, 2.9687e-01, 9.5492e-01, 2.8336e-01, 9.5901e-01,
        2.7043e-01, 9.6274e-01, 2.5806e-01, 9.6613e-01, 2.4623e-01, 9.6921e-01,
        2.3492e-01, 9.7201e-01, 2.2411e-01, 9.7456e-01, 2.1378e-01, 9.7688e-01,
        2.0391e-01, 9.7899e-01, 1.9449e-01, 9.8091e-01, 1.8549e-01, 9.8265e-01,
        1.7689e-01,

In [ ]:
# Get one embedding vector from dataframe
embedding1 = torch.tensor(
    embedding1,
    dtype=torch.float32
)

# Add positional encoding
final_token_bn= embedding1 + pe

#print("Token Embedding:", embedding)
print("Final Transformer Input:", final_token_bn)

#print("Final Shape:", final_token_bn.shape)

Final Transformer Input: tensor([[0.8698, 0.4561, 0.7716,  ..., 1.0686, 0.0663, 1.0673],
        [0.8642, 0.4700, 0.7912,  ..., 1.0541, 0.0629, 1.0481],
        [0.8694, 0.5160, 0.7665,  ..., 1.0706, 0.0505, 1.0295],
        ...,
        [0.8786, 0.4637, 0.7700,  ..., 1.0563, 0.0610, 1.0529],
        [0.8657, 0.4694, 0.7472,  ..., 1.0517, 0.0697, 1.0312],
        [0.8683, 0.4570, 0.7807,  ..., 1.0743, 0.0546, 1.0412]])


In [ ]:
import math
import torch
import torch.nn as nn
d_model=384

class QKV(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q_m = nn.Linear(d_model, d_model)
        self.k_m = nn.Linear(d_model, d_model)
        self.v_m = nn.Linear(d_model, d_model)

    def forward(self, final_token_bn):
        return self.q_m(final_token_bn), self.k_m(final_token_bn), self.v_m(final_token_bn)


class MaskedAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):
        dk = K.shape[-1]

        scores = torch.matmul(Q, K.transpose(-2, -1))   
        scores = scores / math.sqrt(dk)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)   
        mask_output = torch.matmul(attention_weights, V)         
        return mask_output, attention_weights


# ---- usage ----
'''B, T, D = 8, 100, 384

x = torch.randn(B, T, D)                        
mask = torch.tril(torch.ones(T, T))             

qkv = QKV(D)
attn = MaskedAttention()

Q, K, V = qkv(x)
out, weights = attn(Q, K, V, mask)

print(out.shape)       
print(weights.shape)   '''

'B, T, D = 8, 100, 384\n\nx = torch.randn(B, T, D)                        \nmask = torch.tril(torch.ones(T, T))             \n\nqkv = QKV(D)\nattn = MaskedAttention()\n\nQ, K, V = qkv(x)\nout, weights = attn(Q, K, V, mask)\n\nprint(out.shape)       \nprint(weights.shape)   '

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.create_matrix = CreateMatrix(d_model)

        self.attention = SelfAttention()

        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, musk_output,final_output2):

        batch_size, seq_len, d_model = musk_output.shape

        # Generate Q, K, V
        K1, V1 = self.create_matrix(musk_output)
        Q1=self.create_matrix(final_output2)
        # Split into multiple heads
        Q = Q1.view(
            batch_size, seq_len,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        K = K1.view(
            batch_size, seq_len,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        V = V1.view(
            batch_size, seq_len,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        # Apply attention
        output, attention_weights = self.attention(Q, K, V)

        # Concatenate all heads
        output = output.transpose(1, 2).contiguous()

        output = output.view(
            batch_size, seq_len, self.d_model
        )

        # Final linear projection
        output = self.W_o(output)

        return output, attention_weights
print(output)

tensor([[[ 1.3526e+01, -1.4365e+01, -4.0356e+00,  ...,  9.9643e+00,
           9.6108e+00,  7.1202e+00],
         [ 1.6071e+01, -2.1578e+01,  1.8796e+01,  ...,  9.8056e+00,
           9.5077e+00,  9.2220e-01],
         [-1.0947e+01, -1.7408e+01,  2.0157e+01,  ...,  4.0795e+00,
           9.1454e+00,  1.3896e+01],
         [ 1.0924e+01, -8.7397e-03,  1.9122e+01,  ..., -1.0478e+01,
           8.7250e+00,  6.6060e+00],
         [-3.5167e+00,  1.3975e+01, -6.5535e-01,  ..., -1.3372e+01,
           1.8826e+01, -1.8504e+00]],

        [[ 2.0780e+01,  1.2962e+01, -1.8829e+00,  ..., -4.0838e-01,
          -1.3335e+01, -1.4420e+01],
         [ 3.3840e-01,  9.2367e+00, -1.4578e+01,  ...,  4.0517e+00,
          -1.7019e+01,  7.9475e+00],
         [ 1.2186e+01,  3.7013e+01, -4.4942e+00,  ..., -9.1330e+00,
          -1.1715e+01, -1.3481e+01],
         [ 4.7359e+00,  1.8572e+01,  9.2882e-01,  ..., -6.4465e+00,
          -1.7895e+01,  1.6520e+00],
         [-6.6614e+00,  1.9216e+01,  1.4393e+01,  ...

In [ ]:
class AddNorm(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, X, sublayer_output):

        # Residual connection
        output = X + sublayer_output

        # Layer normalization
        output = self.layer_norm(output)

        return output
print(output)

tensor([[[ 1.3526e+01, -1.4365e+01, -4.0356e+00,  ...,  9.9643e+00,
           9.6108e+00,  7.1202e+00],
         [ 1.6071e+01, -2.1578e+01,  1.8796e+01,  ...,  9.8056e+00,
           9.5077e+00,  9.2220e-01],
         [-1.0947e+01, -1.7408e+01,  2.0157e+01,  ...,  4.0795e+00,
           9.1454e+00,  1.3896e+01],
         [ 1.0924e+01, -8.7397e-03,  1.9122e+01,  ..., -1.0478e+01,
           8.7250e+00,  6.6060e+00],
         [-3.5167e+00,  1.3975e+01, -6.5535e-01,  ..., -1.3372e+01,
           1.8826e+01, -1.8504e+00]],

        [[ 2.0780e+01,  1.2962e+01, -1.8829e+00,  ..., -4.0838e-01,
          -1.3335e+01, -1.4420e+01],
         [ 3.3840e-01,  9.2367e+00, -1.4578e+01,  ...,  4.0517e+00,
          -1.7019e+01,  7.9475e+00],
         [ 1.2186e+01,  3.7013e+01, -4.4942e+00,  ..., -9.1330e+00,
          -1.1715e+01, -1.3481e+01],
         [ 4.7359e+00,  1.8572e+01,  9.2882e-01,  ..., -6.4465e+00,
          -1.7895e+01,  1.6520e+00],
         [-6.6614e+00,  1.9216e+01,  1.4393e+01,  ...

In [ ]:
class CrossAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Q projects the decoder; K and V project the encoder
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.attention = SelfAttention()      # your scaled dot-product class
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, decoder_x, encoder_output):

        batch_size, tgt_len, _ = decoder_x.shape       # decoder length
        src_len = encoder_output.shape[1]              # encoder length

        Q = self.W_q(decoder_x)         # [B, tgt_len, d_model]
        K = self.W_k(encoder_output)    # [B, src_len, d_model]
        V = self.W_v(encoder_output)    # [B, src_len, d_model]

        # Split into heads
        Q = Q.view(batch_size, tgt_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)

        # scores: [B, heads, tgt_len, src_len]
        out, attn_weights = self.attention(Q, K, V)

        # Merge heads
        out = out.transpose(1, 2).contiguous().view(batch_size, tgt_len, self.d_model)

        return self.W_o(out), attn_weights
cross_attention = CrossAttention(d_model, num_heads)
cross_output, cross_attention_weights = cross_attention(final_output2, final_output)

print("Cross-attention output:", cross_output)
print("Attention weights:", cross_attention_weights.shape)

Cross-attention output: tensor([[[ 0.0802,  0.0398,  0.1980,  ..., -0.2698,  0.0906, -0.0786],
         [ 0.0329, -0.0196,  0.1955,  ..., -0.2952,  0.0996, -0.1076],
         [ 0.0606,  0.0206,  0.2278,  ..., -0.2437,  0.1235, -0.1304],
         [ 0.0527,  0.0298,  0.1745,  ..., -0.2379,  0.0858, -0.0743],
         [ 0.0975,  0.0064,  0.1531,  ..., -0.2093,  0.1085, -0.1296]],

        [[ 0.2760, -0.0727, -0.1800,  ..., -0.1408, -0.1256,  0.0468],
         [ 0.2569, -0.0566, -0.1621,  ..., -0.1676, -0.1755,  0.0193],
         [ 0.2839, -0.0862, -0.1452,  ..., -0.1668, -0.1300,  0.0274],
         [ 0.2586, -0.0930, -0.1310,  ..., -0.1500, -0.1314,  0.0228],
         [ 0.2872, -0.0606, -0.1151,  ..., -0.1392, -0.1068, -0.0407]]],
       grad_fn=<ViewBackward0>)
Attention weights: torch.Size([2, 12, 5, 5])


In [ ]:
import torch
import torch.nn as nn

class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_output):
        return self.norm(x + self.dropout(sublayer_output))

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, output, d_ff):
        super().__init__()

        self.linear1 = nn.Linear(output, d_ff)

        self.relu = nn.GELU()

        self.linear2 = nn.Linear(d_ff, output)

    def forward(self, X):

        X = self.linear1(X)
        X = self.relu(X)
        X = self.linear2(X)

        return X

In [ ]:
class DecoderLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)

        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_out, causal_mask, src_mask):

        # Masked self-attention
        attn = self.self_attention(x, x, x, causal_mask)
        x = self.norm1(x + self.dropout(attn))

        # Encoder-decoder cross-attention
        cross = self.cross_attention(
            x, encoder_out, encoder_out, src_mask
        )
        x = self.norm2(x + self.dropout(cross))

        # Feed Forward
        ffn = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn))

        return x


In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---------------- Config ----------------
VOCAB_SIZE = 39065
MASK_ID = 39064          # last id reserved as [MASK]
PAD_ID = 0
D_MODEL = 384
NUM_HEADS = 12
D_FF = 4 * D_MODEL
NUM_LAYERS = 4
MAX_LEN = 60
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ---------------- Model parts ----------------
class SelfAttention(nn.Module):
    def forward(self, Q, K, V, mask=None):
        dk = K.shape[-1]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(dk)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        return torch.matmul(weights, V), weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.d_model = d_model

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.attention = SelfAttention()
        self.W_o = nn.Linear(d_model, d_model)

    def split(self, x):
        B, L, _ = x.shape
        return x.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, mask=None):
        B, L, _ = x.shape
        Q, K, V = self.split(self.W_q(x)), self.split(self.W_k(x)), self.split(self.W_v(x))
        out, weights = self.attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_o(out), weights


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_out):
        return self.norm(x + self.dropout(sublayer_out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.act = nn.GELU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.act(self.linear1(x)))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = AddNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.add_norm2 = AddNorm(d_model)

    def forward(self, x, mask=None):
        attn_out, _ = self.mha(x, mask)
        x = self.add_norm1(x, attn_out)
        x = self.add_norm2(x, self.ffn(x))
        return x


class EncoderModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList(
            [EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)]
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_embedding(pos)

        # padding mask: [B, 1, 1, L]
        mask = (input_ids != PAD_ID).unsqueeze(1).unsqueeze(2)

        for layer in self.layers:
            x = layer(x, mask)
        return self.lm_head(x)          # [B, L, vocab]


# ---------------- Data (dummy, replace with yours) ----------------
class MLMDataset(Dataset):
def __init__(self, num_samples=2000, seq_len=60, mask_prob=0.15):
        self.data = torch.randint(1, VOCAB_SIZE - 1, (num_samples, seq_len))
        self.mask_prob = mask_prob

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids = self.data[idx].clone()
        labels = ids.clone()
        masked = torch.rand(ids.shape) < self.mask_prob
        labels[~masked] = -100          # ignored by CrossEntropyLoss
        ids[masked] = MASK_ID
        return ids, labels


loader = DataLoader(MLMDataset(), batch_size=BATCH_SIZE, shuffle=True)

# ---------------- Training ----------------
model = EncoderModel(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, MAX_LEN).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for input_ids, labels in loader:
        input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE)

        logits = model(input_ids)                               # [B, L, V]
        loss = criterion(logits.view(-1, VOCAB_SIZE), labels.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{EPOCHS}  Loss: {total_loss / len(loader):.4f}")

torch.save(model.state_dict(), "encoder_mlm.pt")

Epoch 1/30  Loss: 10.7599
Epoch 2/30  Loss: 10.6988
Epoch 3/30  Loss: 10.6602
Epoch 4/30  Loss: 10.5948
Epoch 5/30  Loss: 10.5318


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader

# ---------------- Data: train + validation ----------------
train_loader = DataLoader(MLMDataset(num_samples=2000), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(MLMDataset(num_samples=500),  batch_size=BATCH_SIZE, shuffle=False)

model = EncoderModel(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, MAX_LEN).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=-100)


# ---------------- Metrics ----------------
def compute_metrics(preds, labels):
    acc = accuracy_score(labels, preds)
    p_m, r_m, f_m, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0)
    return {
        "acc": acc,
        "precision_macro": p_m, "recall_macro": r_m, "f1_macro": f_m,
        "precision_weighted": p_w, "recall_weighted": r_w, "f1_weighted": f_w,
    }


# ---------------- One epoch (train or eval) ----------------
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(train):
        for input_ids, labels in loader:
            input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE)

            logits = model(input_ids)                                   # [B, L, V]
            loss = criterion(logits.view(-1, VOCAB_SIZE), labels.view(-1))

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 10)
                optimizer.step()

            total_loss += loss.item()

            # score only the masked positions
            keep = labels != -100
            all_preds.append(logits.argmax(dim=-1)[keep].cpu())
            all_labels.append(labels[keep].cpu())

    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    metrics = compute_metrics(preds, labels)
    metrics["loss"] = total_loss / len(loader)
    return metrics


# ---------------- Training loop ----------------
for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"  Train | loss {tr['loss']:.4f} | acc {tr['acc']:.4f} | "
          f"P {tr['precision_macro']:.4f} | R {tr['recall_macro']:.4f} | F1 {tr['f1_macro']:.4f}")
    print(f"  Val   | loss {va['loss']:.4f} | acc {va['acc']:.4f} | "
          f"P {va['precision_macro']:.4f} | R {va['recall_macro']:.4f} | F1 {va['f1_macro']:.4f}")

torch.save(model.state_dict(), "encoder_mlm.pt")

Epoch 1/30
  Train | loss 10.7485 | acc 0.0000 | P 0.0000 | R 0.0000 | F1 0.0000
  Val   | loss 10.7750 | acc 0.0000 | P 0.0000 | R 0.0000 | F1 0.0000
Epoch 2/30
  Train | loss 10.7012 | acc 0.0001 | P 0.0000 | R 0.0000 | F1 0.0000
  Val   | loss 10.7737 | acc 0.0000 | P 0.0000 | R 0.0000 | F1 0.0000
Epoch 3/30
  Train | loss 10.6409 | acc 0.0000 | P 0.0000 | R 0.0000 | F1 0.0000
  Val   | loss 10.7701 | acc 0.0000 | P 0.0000 | R 0.0000 | F1 0.0000
Epoch 4/30
  Train | loss 10.5859 | acc 0.0001 | P 0.0000 | R 0.0001 | F1 0.0000
  Val   | loss 10.8028 | acc 0.0002 | P 0.0000 | R 0.0002 | F1 0.0000
Epoch 5/30
  Train | loss 10.5290 | acc 0.0001 | P 0.0000 | R 0.0001 | F1 0.0000
  Val   | loss 10.7865 | acc 0.0000 | P 0.0000 | R 0.0000 | F1 0.0000
Epoch 6/30
  Train | loss 10.4646 | acc 0.0004 | P 0.0000 | R 0.0003 | F1 0.0000
  Val   | loss 10.7924 | acc 0.0000 | P 0.0000 | R 0.0000 | F1 0.0000
Epoch 7/30
  Train | loss 10.4069 | acc 0.0004 | P 0.0000 | R 0.0003 | F1 0.0000
  Val   | los